# 02. Lakáspiaci Árstruktúra és Szegmentáció

**Cél**: A kőbányai eladó lakások szegmentálása építőanyag (tégla vs. panel), műszaki állapot és szobaszám szerint.

---

### 📖 Miért nem létezik egységes ingatlanpiac Kőbányán?
Az ingatlanpiac valójában egymással párhuzamosan létező **részpiacok (szubpiacok)** összessége. 
Ebben a fejezetben számszerűsítjük azokat a konkrét felárakat és diszkontokat, amelyeket a vásárlók a valóságban megfizetnek:
- Mennyivel olcsóbb egy panellakás egy téglánál?
- Mennyit ér a felújított állapot?
- Hogyan alakulnak a szobaszám-prémiumok?

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
display(HTML(render_area_header_html(df=df)))
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Elemzett eladó lakások száma: {len(elado)} db.")

### 1. Panel vs. Tégla Árolló és Építőanyag Szegmentáció

**📌 Mit mutatnak az adatok a panel és tégla összehasonlításakor?**
Az adathalmazban található 788 db téglalakás és 352 db panellakás pontos összevetése:
- **Téglaépítésű lakások**: Medián ár / m² = **1 271 776 Ft/m²** (átlag: 1 268 319 Ft/m²).
- **Panellakások**: Medián ár / m² = **1 100 000 Ft/m²** (átlag: 1 119 865 Ft/m²).
- **A paneldiszkont pontos mértéke: 13.5%!**
  - A téglalakások mediánjához képest a panellakások **négyzetméterenként ~172 ezer forinttal olcsóbbak**.
  - Egy átlagos 53 m²-es lakásméretnél ez **~9.1 millió forintos vételár-különbséget** jelent a tégla javára!

**🔍 Miért olcsóbb a panel?**
A házgyári technológia merev válaszfalai, a gyengébb hangszigetelés és a nagyobb lakósűrűség árcsökkentő tényező. Ugyanakkor a felújított és szigetelt panelek rezsije kiszámítható, így keresett belépő kategóriát alkotnak.

**💡 Vásárlói tanulság**: Aki a legtöbb négyzetmétert keresi a rendelkezésre álló keretből, a panellakásokkal 13.5%-os közvetlen árelőnyt ér el a téglához képest.

In [ ]:
n_panel = int(elado['is_panel'].sum())
n_tegla = int((elado['is_panel'] == 0).sum())
med_panel = elado[elado['is_panel'] == 1]['nm_ar_huf'].median()
med_tegla = elado[elado['is_panel'] == 0]['nm_ar_huf'].median()
tegla_premium_pct = ((med_tegla - med_panel) / med_panel) * 100

kpi_cards = [
    ("Panel Lakások", f"{n_panel:,} db".replace(',', ' '), f"{n_panel/len(elado)*100:.1f}% kínálat", "#ef4444"),
    ("Tégla Lakások", f"{n_tegla:,} db".replace(',', ' '), f"{n_tegla/len(elado)*100:.1f}% kínálat", "#06b6d4"),
    ("Panel Medián Ár/m²", fmt_huf(med_panel), "Fajlagos ár", "#f87171"),
    ("Tégla Medián Ár/m²", fmt_huf(med_tegla), "Fajlagos ár", "#22d3ee"),
    ("Tégla Prémium", f"+{tegla_premium_pct:.1f}%", "Fajlagos árkülönbözet", "#10b981"),
    ("Prémium Kategória (>80M)", f"{(elado['ar_millio_ft'] > 80).sum()} db", "Felső piaci szegmens", "#8b5cf6")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. A Műszaki Állapot Értéknövelő Hatása (Állapot-Prémium)

**📌 Mit mutat a műszaki állapot szerinti bontás?**
A lakások állapota (felújítandó, átlagos, felújított, újszerű, új építésű) szerint vizsgált négyzetméterárak:
- **Felújítandó lakások**: A legalsó ársávban mozognak (~950 ezer – 1.05 millió Ft/m² körül).
- **Felújított / Újszerű lakások**: Átlépik az 1.25 – 1.35 millió Ft/m²-es szintet.
- **Kategóriánkénti ugrás**: A hedonikus regressziós modellünk szerint (NB07) minden egyes minőségi kategórialépés átlagosan **+7.6%-os tiszta árprémiumot** jelent négyzetméterenként!

**💡 Befektetői stratégia**: Az elhanyagolt lakások megvásárlása és minőségi felújítása stabil, forintosítható értéktöbbletet hoz létre.

In [ ]:
elado['Epites_Tipus'] = elado['is_panel'].map({1: 'Panel', 0: 'Tégla / Egyéb'})

fig1 = px.violin(
    elado,
    x='varosresz',
    y='nm_ar_huf',
    color='Epites_Tipus',
    box=True,
    points='all',
    title='Panel vs. Tégla négyzetméterár eloszlások városrészenként',
    labels={'varosresz': 'Városrész', 'nm_ar_huf': 'Ár / m² (HUF)', 'Epites_Tipus': 'Épülettípus'},
    color_discrete_map={'Panel': '#ef4444', 'Tégla / Egyéb': '#06b6d4'},
    template=PLOTLY_TEMPLATE
)
fig1.update_layout(xaxis_tickangle=-30, height=480)
fig1.show()

### 3. Szobaszám és Méretkategória Prémiumok

**📌 Mit látunk a szobaszám szerinti vizsgálatban?**
A kőbányai piacon az átlagos szobaszám **2.4 szoba**, a leggyakoribb kategória a **2.0 szobás lakás** (módusz).
- **1 és 1.5 szobás garzonok**: Fajlagos (m²) áruk a legmagasabb, mivel az alacsonyabb abszolút vételár miatt a fiatalok és befektetők körében a legnagyobb a kereslet.
- **2 és 2.5 szobás lakások**: A leglikvidebb családi méretkategória.
- **3+ szobás lakások**: Alacsonyabb fajlagos m² ár, de magasabb összvételár.

**💡 Tanulság**: A garzonlakások fajlagosan drágábbak, de alacsonyabb tőkével megvásárolhatók.

In [ ]:
# Méretkategóriák képzése
elado['meret_kategoria'] = pd.cut(
    elado['alapterulet_nm'],
    bins=[0, 40, 55, 70, 90, 300],
    labels=['<40 m²', '40-55 m²', '55-70 m²', '70-90 m²', '>90 m²']
)

# Kereszttábla aggregáció
pivot_nmar = elado.pivot_table(
    index='meret_kategoria',
    columns='szobaszam_kategoria',
    values='nm_ar_huf',
    aggfunc='median'
)

fig2 = px.imshow(
    pivot_nmar,
    text_auto='.0f',
    color_continuous_scale='Blues',
    title='Medián Négyzetméterár (Ft/m²) Méret és Szobaszám Mátrixban',
    labels={'x': 'Szobaszám Kategória', 'y': 'Alapterület Kategória', 'color': 'Ár / m²'},
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(height=420)
fig2.show()

# Emeleti árazási hatás
emelet_stat = elado.groupby(['is_foldszint', 'is_zaroszint'])['nm_ar_huf'].median().reset_index()
emelet_stat['Pozicio'] = emelet_stat.apply(
    lambda r: 'Földszint' if r['is_foldszint']==1 else ('Zárószint' if r['is_zaroszint']==1 else 'Közbenső emelet'), axis=1
)
fig3 = px.bar(
    emelet_stat,
    x='Pozicio',
    y='nm_ar_huf',
    color='Pozicio',
    title='Földszint és Zárószint Árdiszkontja a Közbenső Emeletekhez Képest',
    labels={'Pozicio': 'Emeleti Elhelyezkedés', 'nm_ar_huf': 'Medián Ár / m² (HUF)'},
    template=PLOTLY_TEMPLATE
)
fig3.update_layout(height=400, showlegend=False)
fig3.show()

### 4. Interaktív Városrész- és Szegmens Laboratórium

**📌 Mire használható a vezérlőfelület?**
A lenyíló menükkel városrészenként szűrheti a panel és tégla lakások megoszlását.
- Figyelje meg: Újhegyen szinte kizárólag panelek találhatók, míg Óhegyen a téglalakások dominálnak.
- Ez magyarázza a két városrész közötti általános árszint-különbséget is!

In [ ]:
# Interaktív Piaci Szegmentációs Elemző (Plotly updatemenus)
szegmens_vars = [
    ('Epites_Tipus', '1. Épülettípus (Panel vs. Tégla)', '#2563eb'),
    ('allapot_kod', '2. Műszaki Állapot Kategória', '#059669'),
    ('szobaszam_kategoria', '3. Szobaszám Szerinti Kategória', '#7c3aed')
]

fig_szeg = go.Figure()
buttons = []

for i, (col, label, colr) in enumerate(szegmens_vars):
    if col in elado.columns:
        agg = elado.groupby(col, observed=True)['nm_ar_huf'].agg(['count', 'median', 'mean']).reset_index()
        sub_fig = px.bar(
            agg, x=col, y='median', text='count',
            labels={'median': 'Medián Ár/m² (Ft)', col: label}
        )
        tr = sub_fig.data[0]
        tr.visible = (i == 0)
        tr.marker.color = colr
        tr.name = label
        fig_szeg.add_trace(tr)
        
        vis = [j == i for j in range(len(szegmens_vars))]
        buttons.append(dict(
            label=label,
            method='update',
            args=[{'visible': vis}, {'title': f'Medián Négyzetméterár {label} szerint (N={len(elado)})', 'xaxis': {'title': label}}]
        ))

fig_szeg.update_layout(
    title=f'Medián Négyzetméterár {szegmens_vars[0][1]} szerint (N={len(elado)})',
    xaxis_title=szegmens_vars[0][1],
    yaxis_title='Medián Fajlagos Ár (Ft/m²)',
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        direction='down',
        x=0.01, y=0.99, xanchor='left', yanchor='top',
        bgcolor='white', bordercolor='#cbd5e1'
    )],
    template=PLOTLY_TEMPLATE,
    height=450
)
fig_szeg.show()